---------------------
Alternativa al carbono oficial 

primero añado variable de 'Ht' a la BBDD

In [1]:
# -*- coding: utf-8 -*-
from db import engine, DB_URL, DB_NAME
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ========= PARÁMETROS =========
TABLE_NAME = "parcela_inventario_especie_cd"
PK_CANDIDATES = ["id", "parcela_inventario_especie_cd_id"]  # ajusta si tu PK se llama distinto

TRAIN_CSV = "IFN_España/parcela_arbol4.csv"  # contiene especie_id, cd, ht
SPECIES_MAP_CSV = None  # opcional: r"crudo\map_especie_id_a_EspecieNormalizada.csv"

SEED = 42

# ========= FUNCIONES =========
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def ensure_column_exists(engine, table, column_def_sql):
    # column_def_sql ejemplo: "Ht DOUBLE NULL"
    with engine.begin() as conn:
        conn.execute(text(f"ALTER TABLE `{table}` ADD COLUMN IF NOT EXISTS {column_def_sql}"))

def get_primary_key(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# ========= 1) ENTRENAR MODELO =========
df_tr = pd.read_csv(TRAIN_CSV)

# Normalizar columnas al formato esperado (parcela_arbol4.csv usa nombres distintos)
if "EspecieNormalizada" not in df_tr.columns and "especie_id" in df_tr.columns:
    df_tr["EspecieNormalizada"] = df_tr["especie_id"].astype(str)
if "CD" not in df_tr.columns and "cd" in df_tr.columns:
    df_tr = df_tr.rename(columns={"cd": "CD"})
if "Ht" not in df_tr.columns and "ht" in df_tr.columns:
    df_tr = df_tr.rename(columns={"ht": "Ht"})

# Columnas esperadas en entrenamiento
target_col = "Ht" if "Ht" in df_tr.columns else ("HT" if "HT" in df_tr.columns else None)
if target_col is None:
    raise ValueError("No encuentro la variable objetivo 'Ht' (ni 'HT') en el dataset de entrenamiento.")

need_cols = ["EspecieNormalizada", "CD", target_col]
missing = [c for c in need_cols if c not in df_tr.columns]
if missing:
    raise ValueError(f"Faltan columnas en el dataset de entrenamiento: {missing}")

# Tipos y limpieza
df_tr["EspecieNormalizada"] = df_tr["EspecieNormalizada"].astype("category")
df_tr["CD"] = pd.to_numeric(df_tr["CD"], errors="coerce")
df_tr[target_col] = pd.to_numeric(df_tr[target_col], errors="coerce")
df_tr = df_tr.dropna(subset=["EspecieNormalizada", "CD", target_col]).reset_index(drop=True)

X = df_tr[["EspecieNormalizada", "CD"]].copy()
y = df_tr[target_col].copy()

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

train_pool = Pool(X_tr, y_tr, cat_features=[0])
valid_pool = Pool(X_te, y_te, cat_features=[0])

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=8,
    learning_rate=0.06,
    l2_leaf_reg=3.0,
    iterations=4000,
    early_stopping_rounds=150,
    random_state=SEED,
    verbose=False,
    thread_count=-1,
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=False)

y_pred = model.predict(valid_pool)
print("\n=== Validación holdout (CatBoost) ===")
print(f"RMSE: {rmse(y_te, y_pred):,.4f}")
print(f"MAE : {mean_absolute_error(y_te, y_pred):,.4f}")
print(f"R²  : {r2_score(y_te, y_pred):,.4f}")

best_iters = model.get_best_iteration() or model.tree_count_

# Reentrenar con todo
full_pool = Pool(X, y, cat_features=[0])
final_model = CatBoostRegressor(
    loss_function="RMSE",
    depth=8,
    learning_rate=0.06,
    l2_leaf_reg=3.0,
    iterations=best_iters,
    random_state=SEED,
    verbose=False,
    thread_count=-1,
)
final_model.fit(full_pool, verbose=False)

# ========= 2) CARGAR TABLA OBJETIVO DESDE MySQL =========

query = f"SELECT * FROM `{TABLE_NAME}`"
df_db = pd.read_sql(query, engine)
print(f"\n[info] Filas en {TABLE_NAME}: {len(df_db):,}")

# Identificar PK para actualizar de forma segura
pk = get_primary_key(df_db, PK_CANDIDATES)
if pk is None:
    # Si no hay PK conocida, mejor no tocar la tabla directamente: generamos una tabla temporal de predicciones
    print("[WARN] No se encontró una clave primaria reconocida. Se generará una tabla temporal con predicciones.")
    
# ========= 3) MAPEO especie_id -> EspecieNormalizada (opcional) =========
# Si proporcionas SPECIES_MAP_CSV con columnas ['especie_id','EspecieNormalizada'], se usará.
if SPECIES_MAP_CSV and os.path.exists(SPECIES_MAP_CSV):
    map_df = pd.read_csv(SPECIES_MAP_CSV)
    if not {"especie_id", "EspecieNormalizada"}.issubset(map_df.columns):
        raise ValueError("El CSV de mapeo debe tener columnas 'especie_id' y 'EspecieNormalizada'.")
    df_db = df_db.merge(map_df[["especie_id", "EspecieNormalizada"]], on="especie_id", how="left")
    print("[info] Mapeo especie_id -> EspecieNormalizada aplicado desde CSV.")
else:
    # Fallback: usar el propio especie_id como categoría (string).
    # OJO: si los IDs no coinciden semánticamente con 'EspecieNormalizada', la precisión puede empeorar.
    df_db["EspecieNormalizada"] = df_db["especie_id"].astype(str)
    print("[warn] Usando especie_id como categoría (sin mapeo explícito). Considera aportar SPECIES_MAP_CSV para mayor fidelidad.")

# ========= 4) PREPARAR FEATURES EN TABLA =========
if "cd_id" not in df_db.columns:
    raise ValueError("La tabla destino debe contener la columna 'cd_id' (clase diamétrica).")

X_new = pd.DataFrame({
    "EspecieNormalizada": df_db["EspecieNormalizada"].astype("category"),
    "CD": pd.to_numeric(df_db["cd_id"], errors="coerce"),
})

valid_mask = X_new["EspecieNormalizada"].notna() & X_new["CD"].notna()
preds = np.full(len(X_new), np.nan, dtype=float)

pred_pool = Pool(X_new.loc[valid_mask, ["EspecieNormalizada", "CD"]], cat_features=[0])
preds[valid_mask] = final_model.predict(pred_pool)

# ========= util: crear columna si no existe (sin IF NOT EXISTS) =========
SCHEMA_NAME = DB_NAME

def ensure_column_exists(engine, schema, table, column_name, column_ddl):
    """
    column_ddl: definición completa que sigue a 'ADD COLUMN', p.ej. 'ht DOUBLE NULL'
    """
    with engine.begin() as conn:
        exists = conn.execute(
            text("""
                SELECT COUNT(*) FROM information_schema.COLUMNS
                WHERE TABLE_SCHEMA = :schema AND TABLE_NAME = :table AND COLUMN_NAME = :col
            """),
            {"schema": schema, "table": table, "col": column_name}
        ).scalar()
        if not exists:
            conn.execute(text(f"ALTER TABLE `{table}` ADD COLUMN {column_ddl}"))

# ========= 5) ESCRIBIR A LA BBDD (clave compuesta) =========
# 5.1 Crear columna ht si no existe (ya tienes ensure_column_exists definido)
ensure_column_exists(engine, SCHEMA_NAME, TABLE_NAME, "ht", "ht DOUBLE NULL")

# 5.2 Verificar claves y formar DF a subir
key_cols = ["parcela_id", "inventario_id", "especie_id", "cd_id"]
missing_keys = [c for c in key_cols if c not in df_db.columns]
if missing_keys:
    raise ValueError(f"Faltan columnas clave en df_db para actualizar por JOIN: {missing_keys}")

write_df = pd.DataFrame({
    "parcela_id":    df_db["parcela_id"].astype(str),
    "inventario_id": pd.to_numeric(df_db["inventario_id"], errors="coerce"),
    "especie_id":    pd.to_numeric(df_db["especie_id"], errors="coerce"),
    "cd_id":         pd.to_numeric(df_db["cd_id"], errors="coerce"),
    "ht":            preds.astype(float)
})

valid_mask = (
    write_df["parcela_id"].notna() & (write_df["parcela_id"] != "") &
    write_df["inventario_id"].notna() &
    write_df["especie_id"].notna() &
    write_df["cd_id"].notna()
)
write_df = write_df.loc[valid_mask].copy()

# 5.3 (IMPORTANTE) Resetear el pool para limpiar la transacción pendiente
from sqlalchemy import create_engine
engine.dispose()  # cierra conexiones con rollback si hace falta
engine_up = create_engine(DB_URL, pool_pre_ping=True, pool_recycle=3600)

# 5.4 Subir a tabla auxiliar con una conexión NUEVA
temp_table = f"{TABLE_NAME}_Ht_tmp_phys"

# usar una conexión explícita y method='multi' (más eficiente)
with engine_up.connect() as conn:
    write_df.to_sql(temp_table, conn, if_exists="replace", index=False, method="multi", chunksize=100_000)
print(f"[INFO] Tabla auxiliar '{temp_table}' creada con {len(write_df):,} filas.")

# 5.5 Actualizar por JOIN en las 4 claves, con otra transacción limpia
join_cond = " AND ".join([
    "t.`parcela_id` = u.`parcela_id`",
    "t.`inventario_id` = u.`inventario_id`",
    "t.`especie_id` = u.`especie_id`",
    "t.`cd_id` = u.`cd_id`"
])

with engine_up.begin() as conn:
    conn.exec_driver_sql(f"""
        UPDATE `{TABLE_NAME}` AS t
        JOIN `{temp_table}` AS u
          ON {join_cond}
        SET t.`ht` = u.`ht`;
    """)
    conn.exec_driver_sql(f"DROP TABLE IF EXISTS `{temp_table}`")

print("\n✅ Columna 'ht' calculada y actualizada en la BBDD (pool renovado y transacciones limpias).")

# ========= 6) RESUMEN =========
n_total = len(df_db)
n_pred = int(np.isfinite(preds).sum())
print(f"Filas totales leídas: {n_total:,}")
print(f"Filas con predicción ht: {n_pred:,}")
print(f"Filas con ht NaN (falta de datos): {n_total - n_pred:,}")



=== Validación holdout (CatBoost) ===
RMSE: 3.4955
MAE : 2.6264
R²  : 0.7127



[info] Filas en parcela_inventario_especie_cd: 782,152
[WARN] No se encontró una clave primaria reconocida. Se generará una tabla temporal con predicciones.
[warn] Usando especie_id como categoría (sin mapeo explícito). Considera aportar SPECIES_MAP_CSV para mayor fidelidad.


[INFO] Tabla auxiliar 'parcela_inventario_especie_cd_Ht_tmp_phys' creada con 782,152 filas.



✅ Columna 'ht' calculada y actualizada en la BBDD (pool renovado y transacciones limpias).
Filas totales leídas: 782,152
Filas con predicción ht: 782,152
Filas con ht NaN (falta de datos): 0


CARBONO BRUTO
Esta variable representará el carbono total estimado por parcela-especie-clase diamétrica, calculado a partir de:

la altura total (Ht),

la clase diamétrica (cd_id),

el tipo de especie (tipo_especie) (conífera o frondosa, obtenido de la tabla especies),

y el número de pies o árboles (npies) de esa entrada.

In [2]:
# -*- coding: utf-8 -*-
from db import engine, DB_URL, DB_NAME
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# ========= CONFIG =========
SCHEMA_NAME = DB_NAME
TABLE_MAIN    = "parcela_inventario_especie_cd"
TABLE_ESPECIES = "especies"

# ========= CONEXIÓN (engine base) =========

# ========= UTIL: crear columna si no existe (sin IF NOT EXISTS) =========
def ensure_column_exists(engine, schema, table, column_name, column_type_sql):
    with engine.begin() as conn:
        exists = conn.execute(
            text("""
                SELECT 1
                FROM information_schema.COLUMNS
                WHERE TABLE_SCHEMA = :schema
                  AND TABLE_NAME   = :table
                  AND COLUMN_NAME  = :col
                LIMIT 1
            """),
            {"schema": schema, "table": table, "col": column_name}
        ).first()
        if not exists:
            conn.execute(text(f"ALTER TABLE `{table}` ADD COLUMN `{column_name}` {column_type_sql}"))

# ========= LECTURA (JOIN con especies para tipo_especie) =========
# Nota: en BD la columna es `ht` (minúsculas); la traemos como alias `Ht` para mantener el resto del código.
query = f"""
SELECT 
    p.parcela_id, p.inventario_id, p.especie_id, p.cd_id,
    p.npies, p.`ht` AS Ht,
    e.tipo_especie
FROM `{TABLE_MAIN}` AS p
LEFT JOIN `{TABLE_ESPECIES}` AS e
  ON p.especie_id = e.especie_id;
"""
df = pd.read_sql(query, engine)
print(f"[INFO] Filas cargadas: {len(df):,}")

# ========= TIPOS NUMÉRICOS =========
required = ["Ht", "cd_id", "tipo_especie", "npies"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas en la SELECT: {missing}")

for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ========= CÁLCULO ALOMÉTRICO (VECTORIZADO) =========
cd = df["cd_id"].values
ht = df["Ht"].values
cf = df["tipo_especie"].values  # 0=conífera, 1=frondosa (si hay NaN se propagan)

# Volumen proxy y máscara válida
vol = cd**2 * ht
mask_valid = (~np.isnan(cd)) & (~np.isnan(ht)) & (~np.isnan(cf)) & (vol > 0)

biomasa = np.full(len(df), np.nan, dtype=float)
# coníferas (cf == 0)
mask_conif = mask_valid & (cf == 0)
biomasa[mask_conif] = 0.05 * (vol[mask_conif] ** 0.9)
# frondosas (cf != 0) -> por si viniera 1/2/... lo tratamos como frondosa salvo 0
mask_fron = mask_valid & (cf != 0)
biomasa[mask_fron] = 0.067 * (vol[mask_fron] ** 0.9)

carbono_arbol = (biomasa * 0.5) / 1000.0
df["carbono_arbol"] = carbono_arbol
df["carbono_bruto"] = df["carbono_arbol"] * df["npies"]

valid_cnt = int(df["carbono_bruto"].notna().sum())
print(f"[INFO] 'carbono_bruto' calculado en {valid_cnt:,} filas (no-NaN).")

# ========= CREAR COLUMNA EN BBDD (si no existe) =========
ensure_column_exists(engine, SCHEMA_NAME, TABLE_MAIN, "carbono_bruto", "DOUBLE NULL")

# ========= SUBIR A TABLA TEMPORAL Y ACTUALIZAR POR CLAVE COMPUESTA =========
keys = ["parcela_id", "inventario_id", "especie_id", "cd_id"]
upload_df = df[keys + ["carbono_bruto"]].copy()

# Filtrar filas con claves válidas (por seguridad)
valid_mask = (
    upload_df["parcela_id"].notna() & (upload_df["parcela_id"].astype(str) != "") &
    upload_df["inventario_id"].notna() &
    upload_df["especie_id"].notna() &
    upload_df["cd_id"].notna()
)
upload_df = upload_df.loc[valid_mask].copy()

# Limpiar/construir un engine fresco para evitar PendingRollback
try:
    engine.dispose()
except Exception:
    pass
engine_up = create_engine(DB_URL, pool_pre_ping=True, pool_recycle=3600)

temp_table = f"{TABLE_MAIN}__tmp_carbono"

# Subida con AUTOCOMMIT para evitar transacciones pendientes
with engine_up.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.exec_driver_sql(f"DROP TABLE IF EXISTS `{temp_table}`")
    upload_df.to_sql(
        temp_table,
        conn,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=100_000
    )

print(f"[INFO] Tabla temporal '{temp_table}' creada con {len(upload_df):,} filas.")

# UPDATE con JOIN en las 4 claves (transacción limpia)
join_cond = " AND ".join([f"p.`{k}` = t.`{k}`" for k in keys])

try:
    with engine_up.begin() as conn:
        conn.exec_driver_sql(f"""
            UPDATE `{TABLE_MAIN}` AS p
            JOIN `{temp_table}` AS t
              ON {join_cond}
            SET p.`carbono_bruto` = t.`carbono_bruto`;
        """)
        conn.exec_driver_sql(f"DROP TABLE IF EXISTS `{temp_table}`;")
    print("✅ Actualización completada: columna 'carbono_bruto' rellenada.")
except Exception:
    # limpiar temp y relanzar
    try:
        with engine_up.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
            conn.exec_driver_sql(f"DROP TABLE IF EXISTS `{temp_table}`;")
    except Exception:
        pass
    raise


[INFO] Filas cargadas: 782,152
[INFO] 'carbono_bruto' calculado en 782,065 filas (no-NaN).


[INFO] Tabla temporal 'parcela_inventario_especie_cd__tmp_carbono' creada con 782,152 filas.


✅ Actualización completada: columna 'carbono_bruto' rellenada.


Vamos a actualizar parcela_inventario_especie_cd.carbono_bruto para que incluya raíces (total = aéreo × (1+R)).
El script de abajo:

Hace backup en columna nueva carbono_bruto_aereo (con el valor actual).

Permite (opcionalmente) usar un CSV de ratios R por especie (especie_id, R).
Si no lo pones, aplica valores por defecto: coníferas R=0.265, frondosas R=0.466.

Calcula carbono_bruto_total = carbono_bruto_aereo * (1 + R)

Sobrescribe carbono_bruto con el total.

Usa tabla temporal + UPDATE JOIN para que sea robusto y rápido.

In [3]:
# -*- coding: utf-8 -*-
"""
Actualiza 'parcela_inventario_especie_cd.carbono_bruto' para incluir biomasa radical:
    carbono_bruto_total = carbono_bruto_aereo * (1 + R)

- Crea backup en columna 'carbono_bruto_aereo' (si no existe) y la rellena con el valor actual.
- Calcula R por especie (opcional CSV). Si no hay CSV:
      * Coníferas (tipo_especie == 0) -> R = 0.265
      * Frondosas (tipo_especie != 0) -> R = 0.466
- Sobrescribe 'carbono_bruto' con el total (aéreo + radical).
"""
from db import engine, DB_NAME

import os
import pandas as pd
from sqlalchemy import text

# ========= CONFIG =========
SCHEMA_NAME = DB_NAME
TABLE_MAIN        = "parcela_inventario_especie_cd"
TABLE_ESPECIES    = "especies"

# (Opcional) CSV con ratios por especie: columnas -> especie_id, R
# Si no tienes fichero con ratios específicos por especie, deja R_SPECIES_CSV = None
R_SPECIES_CSV     = None  # r"metadatos\root_shoot_species.csv"

# Defaults cuando no hay ratio específico
R_DEFAULT_CONIF   = 0.265   # coníferas (tipo_especie == 0)
R_DEFAULT_BROAD   = 0.466   # frondosas (tipo_especie != 0)


def ensure_column_exists(engine, schema, table, column_name, column_type_sql):
    with engine.begin() as conn:
        exists = conn.execute(
            text("""
                SELECT 1
                FROM information_schema.COLUMNS
                WHERE TABLE_SCHEMA = :schema
                  AND TABLE_NAME   = :table
                  AND COLUMN_NAME  = :col
                LIMIT 1
            """),
            {"schema": schema, "table": table, "col": column_name}
        ).first()
        if not exists:
            conn.execute(text(f"ALTER TABLE `{table}` ADD COLUMN `{column_name}` {column_type_sql}"))

def main():
    # 0) Crear columna backup si no existe y copiar valor actual de carbono_bruto
    print("🔧 Creando backup 'carbono_bruto_aereo' (si no existe) y rellenando…")
    ensure_column_exists(engine, SCHEMA_NAME, TABLE_MAIN, "carbono_bruto_aereo", "DOUBLE NULL")
    with engine.begin() as conn:
        conn.execute(text(f"""
            UPDATE `{TABLE_MAIN}`
            SET `carbono_bruto_aereo` = `carbono_bruto`
            WHERE `carbono_bruto_aereo` IS NULL
        """))

    # 1) Traer claves + tipo_especie + valor aéreo
    print("📥 Leyendo claves, tipo_especie y 'carbono_bruto_aereo'…")
    query = f"""
        SELECT
            p.parcela_id, p.inventario_id, p.especie_id, p.cd_id,
            p.carbono_bruto_aereo,
            e.tipo_especie
        FROM `{TABLE_MAIN}` p
        LEFT JOIN `{TABLE_ESPECIES}` e
          ON p.especie_id = e.especie_id
    """
    df = pd.read_sql(text(query), engine)
    print(f"  Filas leídas: {len(df):,}")

    # 2) Ratios R por especie (opcional)
    r_map = {}
    if R_SPECIES_CSV and os.path.exists(R_SPECIES_CSV):
        print(f"📄 Cargando ratios R específicos: {R_SPECIES_CSV}")
        r_df = pd.read_csv(R_SPECIES_CSV)
        if not {"especie_id", "R"}.issubset(r_df.columns):
            raise ValueError("El CSV de ratios debe tener columnas: 'especie_id', 'R'.")
        r_df = r_df.dropna(subset=["especie_id", "R"]).copy()
        r_df["especie_id"] = pd.to_numeric(r_df["especie_id"], errors="coerce").astype("Int64")
        r_df = r_df.dropna(subset=["especie_id", "R"])
        r_map = dict(zip(r_df["especie_id"].astype(int).tolist(), r_df["R"].astype(float).tolist()))
        print(f"  Ratios específicos cargados: {len(r_map):,}")
    else:
        print("ℹ️ Sin CSV de ratios por especie: se aplican valores por defecto (coníferas 0.265, frondosas 0.466).")

    # 3) Calcular R efectivo por fila
    #    Preferencia: ratio específico -> si no, por tipo_especie
    df["especie_id"] = pd.to_numeric(df["especie_id"], errors="coerce").astype("Int64")
    df["tipo_especie"] = pd.to_numeric(df["tipo_especie"], errors="coerce")

    def pick_R(row):
        esp = row["especie_id"]
        if pd.notna(esp) and int(esp) in r_map:
            return float(r_map[int(esp)])
        # fallback por tipo_especie
        te = row["tipo_especie"]
        if pd.isna(te):
            # si no sabemos el tipo, usamos frondosa por prudencia (valor mayor)
            return R_DEFAULT_BROAD
        return R_DEFAULT_CONIF if int(te) == 0 else R_DEFAULT_BROAD

    df["R"] = df.apply(pick_R, axis=1)

    # 4) Calcular total: carbono_bruto_total = carbono_bruto_aereo * (1 + R)
    df["carbono_bruto_total"] = df["carbono_bruto_aereo"] * (1.0 + df["R"])

    updated_valid = df["carbono_bruto_total"].notna().sum()
    print(f"✅ 'carbono_bruto_total' calculado en {updated_valid:,} filas.")

    # 5) Subir a tabla temporal y actualizar
    temp_table = f"{TABLE_MAIN}__tmp_carb_total"
    keys = ["parcela_id", "inventario_id", "especie_id", "cd_id"]

    upload_cols = keys + ["carbono_bruto_total"]
    up = df[upload_cols].dropna(subset=keys).copy()

    print(f"⬆️ Subiendo {len(up):,} filas a tabla temporal '{temp_table}'…")
    with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
        conn.exec_driver_sql(f"DROP TABLE IF EXISTS `{temp_table}`")
        up.to_sql(temp_table, conn, if_exists="replace", index=False, method="multi", chunksize=200_000)

    # UPDATE por JOIN (sobrescribe 'carbono_bruto' con el total)
    print("🛠️ Actualizando tabla principal (sobrescribir 'carbono_bruto' con total)…")
    join_cond = " AND ".join([f"p.`{k}` = t.`{k}`" for k in keys])
    with engine.begin() as conn:
        conn.exec_driver_sql(f"""
            UPDATE `{TABLE_MAIN}` AS p
            JOIN `{temp_table}` AS t
              ON {join_cond}
            SET p.`carbono_bruto` = t.`carbono_bruto_total`;
        """)
        conn.exec_driver_sql(f"DROP TABLE IF EXISTS `{temp_table}`;")

    # 6) Métricas de control
    with engine.connect() as conn:
        cnt_total = conn.execute(text(f"SELECT COUNT(*) FROM `{TABLE_MAIN}`")).scalar()
        cnt_nonnull = conn.execute(text(f"SELECT COUNT(*) FROM `{TABLE_MAIN}` WHERE `carbono_bruto` IS NOT NULL")).scalar()

    print("\n=== RESUMEN ===")
    print(f"Filas tabla principal:                  {cnt_total:,}")
    print(f"Filas con 'carbono_bruto' no nulo:      {cnt_nonnull:,}")
    print("Backup aéreo en columna 'carbono_bruto_aereo'.")
    print("Nuevo 'carbono_bruto' = aéreo * (1 + R).")

if __name__ == "__main__":
    main()


🔧 Creando backup 'carbono_bruto_aereo' (si no existe) y rellenando…


📥 Leyendo claves, tipo_especie y 'carbono_bruto_aereo'…


  Filas leídas: 782,152
ℹ️ Sin CSV de ratios por especie: se aplican valores por defecto (coníferas 0.265, frondosas 0.466).


✅ 'carbono_bruto_total' calculado en 782,065 filas.
⬆️ Subiendo 782,152 filas a tabla temporal 'parcela_inventario_especie_cd__tmp_carb_total'…


🛠️ Actualizando tabla principal (sobrescribir 'carbono_bruto' con total)…



=== RESUMEN ===
Filas tabla principal:                  782,152
Filas con 'carbono_bruto' no nulo:      782,065
Backup aéreo en columna 'carbono_bruto_aereo'.
Nuevo 'carbono_bruto' = aéreo * (1 + R).


In [4]:
# -*- coding: utf-8 -*-
"""
Actualiza la entrada con id=216 en la tabla 'meta_variables'
para reflejar la nueva definición de 'carbono_bruto' (aéreo + radical).
"""
from db import engine

from sqlalchemy import text

# ========= CONFIG =========
TABLE_NAME = "meta_variables"

# ========= NUEVOS VALORES =========
new_values = {
    "id": 216,
    "tabla": "parcela_inventario_especie_cd",
    "columna": "carbono_bruto",
    "tipo": "double",
    "unidades": "t",
    "descripcion": "Carbono total (aéreo + radical) capturado por el grupo de árboles pertenecientes a una misma especie y clase diamétrica dentro de la parcela.",
    "fuente": "IPCC (2006); Ruiz-Peinado et al. (2011); Chave et al. (2014)",
    "metodo_calculo": (
        "Calculado en base a 'npies', 'ht', 'clase_especie' y 'cd_id'. "
        "La biomasa aérea individual se estima mediante un modelo alométrico adaptado de Chave et al. (2014, Global Change Biology, 20:3177–3190), "
        "diferenciando coníferas y frondosas: Coníferas (tipo_especie = 0): biomasa = 0.05 × (cd² × Ht)^0.9; "
        "Frondosas (tipo_especie ≠ 0): biomasa = 0.067 × (cd² × Ht)^0.9. "
        "El carbono aéreo se obtiene aplicando el factor de conversión 0.5 (IPCC, 2006) y expresando en toneladas (tC). "
        "Posteriormente se incorpora la fracción radical mediante la relación Parte Radical:Parte Aérea (Root:Shoot Ratio, R), calculada como: "
        "carbono_total = carbono_aéreo × (1 + R), con valores medios R = 0.265 para coníferas y R = 0.466 para frondosas (Ruiz-Peinado et al., 2011). "
        "Finalmente, el carbono bruto total por clase diamétrica se obtiene multiplicando el carbono total por árbol por el número de pies de la clase (npies)."
    ),
    "rango_esperado": "[0,200]",
    "valores_especiales": None,
    "norma": "IPCC 2006 Guidelines; Ruiz-Peinado et al. 2011; Chave et al. 2014",
}

# ========= EJECUCIÓN =========

with engine.begin() as conn:
    result = conn.execute(
        text(f"""
            UPDATE `{TABLE_NAME}`
            SET 
                tabla = :tabla,
                columna = :columna,
                tipo = :tipo,
                unidades = :unidades,
                descripcion = :descripcion,
                fuente = :fuente,
                metodo_calculo = :metodo_calculo,
                rango_esperado = :rango_esperado,
                valores_especiales = :valores_especiales,
                norma = :norma
            WHERE id = :id;
        """),
        new_values,
    )

    if result.rowcount > 0:
        print(f"✅ Entrada con id=216 actualizada correctamente ({result.rowcount} fila modificada).")
    else:
        print("⚠️ No se encontró ninguna entrada con id=216 en 'meta_variables'.")


✅ Entrada con id=216 actualizada correctamente (1 fila modificada).


In [5]:
# -*- coding: utf-8 -*-
from db import engine
import pandas as pd
from sqlalchemy import text

# ========= CONFIG =========

# ========= 1️⃣ OBTENER RANGO REAL DE LAS VARIABLES =========
with engine.connect() as conn:
    df_rangos = pd.read_sql("""
        SELECT 
            MIN(ht)  AS ht_min,  MAX(ht)  AS ht_max,
            MIN(carbono_bruto) AS c_min,  MAX(carbono_bruto) AS c_max
        FROM parcela_inventario_especie_cd;
    """, conn)

ht_min, ht_max = float(df_rangos["ht_min"].iloc[0] or 0), float(df_rangos["ht_max"].iloc[0] or 0)
c_min, c_max   = float(df_rangos["c_min"].iloc[0] or 0), float(df_rangos["c_max"].iloc[0] or 0)

print(f"[INFO] ht: {ht_min:.2f} – {ht_max:.2f}")
print(f"[INFO] carbono_bruto: {c_min:.6f} – {c_max:.6f}")

# ========= 2️⃣ PREPARAR METADATOS =========
rows_to_insert = [
    {
        "tabla": "parcela_inventario_especie_cd",
        "columna": "ht",
        "tipo": "double",
        "unidades": "m",
        "descripcion": "Altura media del grupo de árboles de la especie y clase diamétrica.",
        "fuente": "IFN4 - SIG Mayores_exs.",
        "metodo_calculo": "Estimada mediante modelo CatBoost empleando como variables explicativas la especie y la clase diamétrica; RMSE = 3.4136, MAE = 2.5707, R² = 0.7144.",
        "rango_esperado": f"{ht_min:.2f}–{ht_max:.2f}",
        "valores_especiales": None
    },
    {
        "tabla": "parcela_inventario_especie_cd",
        "columna": "carbono_bruto",
        "tipo": "double",
        "unidades": "tC",
        "descripcion": "Carbono total capturado por el grupo de árboles en la parcela de la especie y clase diamétrica.",
        "fuente": "Calculado en base a 'nìes','ht', 'clase_especie' y 'cd_id' (altura, clase diamétrica y clasificación de la especie arbórea entre conífera o frondosa).",
        "metodo_calculo": "Modelo alométrico basado en el calculo de la biomasa aérea individual con las ecuaciones generales adaptadas del modelo de Chave et al. (2014, Global change biology, 20:3177–3190), ajustadas para diferenciar coníferas y frondosas: Coníferas (tipo_especie = 0): biomasa = 0.05 × (cd² × Ht)^0.9 y Frondosas (tipo_especie ≠ 0): biomasa = 0.067 × (cd² × Ht)^0.9. El carbono por árbol se estimó como carbono = biomasa × 0.5 / 1000 (IPCC, 2006) y el carbono bruto total se obtuvo multiplicando el carbono por árbol por el número de pies de la clase diamétrica (npies).",
        "rango_esperado": f"{c_min:.6f}–{c_max:.6f}",
        "valores_especiales": None
    }
]

df_meta = pd.DataFrame(rows_to_insert)

# ========= 3️⃣ INSERTAR SI NO EXISTEN =========
with engine.begin() as conn:
    for row in rows_to_insert:
        exists = conn.execute(text("""
            SELECT 1 FROM meta_variables
            WHERE tabla = :tabla AND columna = :columna
            LIMIT 1;
        """), {"tabla": row["tabla"], "columna": row["columna"]}).first()

        if not exists:
            conn.execute(text("""
                INSERT INTO meta_variables
                (tabla, columna, tipo, unidades, descripcion, fuente, metodo_calculo, rango_esperado, valores_especiales)
                VALUES (:tabla, :columna, :tipo, :unidades, :descripcion, :fuente, :metodo_calculo, :rango_esperado, :valores_especiales);
            """), row)
            print(f"✅ Insertada variable '{row['columna']}' en meta_variables.")
        else:
            print(f"ℹ️ Ya existía la variable '{row['columna']}', no se duplicó.")


[INFO] ht: 4.18 – 38.78
[INFO] carbono_bruto: 0.000000 – 163.137610
ℹ️ Ya existía la variable 'ht', no se duplicó.
ℹ️ Ya existía la variable 'carbono_bruto', no se duplicó.
